In [1]:
import os, subprocess, sys

# 1. Clone your Movio repository (or upload as a Kaggle dataset)
!git clone https://github.com/tripathiji1312/movio-tts.git movio
%cd movio

# 2. Install dependencies
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate soundfile librosa jieba ctranslate2 fastapi uvicorn websockets pydantic pyyaml torchdiffeq x_transformers pypinyin vocos reddis huggingface_hub
!pip install -q f5-tts ai4bharat-transliteration

# 3. Check GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Cloning into 'movio'...
remote: Enumerating objects: 472, done.
remote: Counting objects: 100% (472/472), done.
remote: Compressing objects: 100% (323/323), done.
remote: Total 472 (delta 211), reused 390 (delta 129), pack-reused 0 (from 0)
Receiving objects: 100% (472/472), 891.50 KiB | 22.86 MiB/s, done.
Resolving deltas: 100% (211/211), done.
/kaggle/working/movio
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 79.7 MB/s eta 0:00:00:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 46.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00
  Preparing m

In [2]:
import yaml

with open("config/settings.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Enable GPU acceleration
cfg["stage_c"]["indicf5"]["device"] = "cuda"
cfg["stage_c"]["indicf5"]["num_flow_steps"] = 16

with open("config/settings.yaml", "w") as f:
    yaml.safe_dump(cfg, f)

print("Config set to CUDA.")

Config set to CUDA.


In [3]:
import os

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

from huggingface_hub import whoami
print('authenticated as:', whoami()['name'])

authenticated as: tripathiji1312


In [ ]:
import subprocess, time, re

# 1. Download cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 2. Start Movio server in the background
server_proc = subprocess.Popen([sys.executable, "-m", "movio"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Movio server starting on port 8000...")
time.sleep(5)

# 3. Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# 4. Extract and print public URL
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

print("\n" + "="*60)
print(f"🚀 LIVE GPU WEB UI & API: {tunnel_url}")
print(f"📡 WebSocket Stream URL : {tunnel_url.replace('https://', 'wss://')}/tts/stream")
print("="*60 + "\n")

# Keep the cell alive and stream server logs
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="")
except KeyboardInterrupt:
    server_proc.terminate()
    tunnel_proc.terminate()

Movio server starting on port 8000...

🚀 LIVE GPU WEB UI & API: https://wayne-symantec-aging-advantage.trycloudflare.com
📡 WebSocket Stream URL : wss://wayne-symantec-aging-advantage.trycloudflare.com/tts/stream

INFO:     Started server process [165]
INFO:     Waiting for application startup.
INFO:movio.textnorm.normalizer:WFST normalizer unavailable; using domain rules only
INFO:movio.router.lid:fasttext available; IndicLID fasttext path possible
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid esc